# Beatrix — early-life curriculum
Her school notebook. Plan of record: `claude-mind/history/plans/2026-08-15_early_life_curriculum.md`.

Nine stages, ~8.8B tokens, resume-first: open this notebook any day, run **Cell 1** then **Cell 3** with whatever hours you have, and walk away. Stage boundaries fire their own instruments. Cell 2 runs ONCE (baseline, both cores). Cell 4 renders growth anytime.

Laws wired in: no chat template / no identity in any training row · causal-test holdout families never reach a training row (bAbI 16+19, depth-5 chains, 3-digit subtraction) · every stage vals on the same fineweb-2013 holdout (gauge continuity) · probes carry the stage-specific measurement.

In [ ]:
%pip install -q "geolip-alephllm @ git+https://github.com/AbstractEyes/alephllm@259bf2ec5b7a85c837122add9b703938484a04e3" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@7e1baba9c0bb1cc38815e4424d969b80ae5a315d"
import os, torch
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # mid-train download
# bars are the tqdm instances that collided with the training bar's
# write path (step-59,000 abort); the trainer is also hardened, this
# removes the source
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
assert torch.cuda.is_available(), "GPU runtime required"
import geolip.alephllm as al
print("geolip", al.__version__, "·", torch.cuda.get_device_name(0))

## Cell 2 — instrument gate (run ONCE, before any training)
Probes P0–P8 on **both** cores — the locked 51,882 and the annealed 58,664 — so every growth curve has a time-zero. Ships the baseline report to the training repo. The head surgery (fold the fossilized gamma gate into W_s; verified semantic no-op, max|diff| 2.4e-07) is applied to the eval copies here and to the live run in Cell 3.

In [ ]:
import json, torch
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, HfApi
from geolip.alephllm.presets import get_preset
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.data import build_tokenizer
from geolip.alephllm.data import curriculum as C
from geolip.alephllm.train import probes

REPO = "AbstractPhil/alephllm-mini-beatrix-training"
# STEPS = None -> probe the NEWEST checkpoint on the hub (numeric step
# compare — the post-rewind current core, never a stale hardcode).
# Set STEPS = [51882, 58664] to re-measure the historical t0 pair.
STEPS = None
import re as _re
from huggingface_hub import HfApi as _HfApi
if STEPS is None:
    _cks = sorted(int(m.group(1)) for f in _HfApi().list_repo_files(REPO)
                  for m in [_re.search(r"step_(\d+)\.safetensors$", f)] if m
                  if "/fp8/" not in f)
    STEPS = [_cks[-1]]
    print(f"probing NEWEST checkpoint: step {STEPS[0]:,}")
preset = get_preset("mini-beatrix-1")
tok = build_tokenizer(preset.model.tokenizer)
baseline = {}
for step in STEPS:
    ck = hf_hub_download(REPO, f"mini-beatrix-1/checkpoints/step_{step:08d}.safetensors")
    m = AlephLM(preset.model); m.load_state_dict(load_file(ck)); m.cuda().eval()
    fold = C.fold_head_gate(m)          # no-op when gamma already 1.0
    res = probes.run_all(m, tok, "cuda")
    baseline[str(step)] = {"probes": res, "head_fold": fold}
    print(f"=== step {step:,} (gamma was {fold['gamma_before']:.6f}) ===")
    print(probes.report(res)); print()
    del m; torch.cuda.empty_cache()
name = ("baseline.json" if STEPS == [51882, 58664]
        else f"gate_step{STEPS[-1]}.json")
with open(name, "w", encoding="utf-8") as f:
    json.dump(baseline, f, indent=1, default=float)
HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=name,
    repo_id=REPO, path_in_repo=f"mini-beatrix-1/reports/curriculum/{name}",
    commit_message="curriculum baseline: probes P0-P8 on both cores at t=0")
print(f"gate report shipped -> reports/curriculum/{name}")

## Cell 2b — rewind (one-time, guarded)
Rolls the run back to a stage boundary and re-plans from there with the corrected mixes. Set `CONFIRM = True` to arm it.

Default target is **step 66,803 (S2 end)**: the measured assessment cleared S1/S2 (fineweb 1.130 → 1.144, probes up) and convicted S3 alone — fineweb → 1.705, grounding/difference/articulation down, and the rule-chain template leaking into her prose (*"Water boils when they are boiling"*). What S3 bought — logic 0.818 → 1.000 including the held-out depth-5 chains — is re-earnable from the fixed mix.

From geolip 0.6.4 on, every boundary also archives `resume/boundary_<stage>_step<N>.pt` (weights + optimizer + stream), so later rewinds restore *exactly*. This one predates those archives, so it restores weights with a fresh optimizer and says so.

In [ ]:
# ONE-TIME REWIND. Set CONFIRM = True to run it; it DISCARDS training
# after ROLLBACK_STEP and re-plans from ROLLBACK_STAGE with the current
# (corrected) mixes. Default target: the S2 boundary, because the
# measured assessment cleared S1/S2 (fineweb 1.130 -> 1.144 flat, probes
# up) and convicted only S3 (fineweb -> 1.705, p0/p2/p8 down, the
# rulechain template leaking into her prose).
CONFIRM = False
ROLLBACK_STEP = 66803
ROLLBACK_STAGE = "curriculum_s3"

from geolip.alephllm import prepare
from geolip.alephllm.data import curriculum as C

if not CONFIRM:
    print("rollback NOT run — set CONFIRM = True to rewind to "
          f"step {ROLLBACK_STEP:,} and replan from {ROLLBACK_STAGE}")
else:
    run = prepare("mini-beatrix-1", hf_token=HF_TOKEN)
    C.append_curriculum_phases(run.manifest)
    if float(run.raw_model.head.gamma.item()) != 1.0:
        C.fold_head_gate(run.raw_model)
    info = C.rollback_to(run, ROLLBACK_STEP, ROLLBACK_STAGE,
                         repo="AbstractPhil/alephllm-mini-beatrix-training")
    # persist the rewind so Cell 3 resumes from it (prepare() reads the
    # hub; an in-memory rewind alone would be undone by the next session)
    run._checkpoint(boundary="rollback")
    print(run.manifest.summary())
    if not info["exact"]:
        print("\nNOTE: fresh optimizer state (this boundary predates the "
              "resume archives) — flag any arm trained from here.")

## Cell 3 — the school day (re-run every session)
Resumes wherever she is. Each `train()` call is capped at the current stage's remaining tokens, so sessions stop **exactly at boundaries**; the boundary fires probes + toggle ledger + head-election gauge and ships the report. Set `MAX_HOURS` to your session budget.

In [ ]:
import json, time, torch
from huggingface_hub import HfApi
from geolip.alephllm import prepare
from geolip.alephllm.data import curriculum as C
from geolip.alephllm.data.streams import build_stream
from geolip.alephllm.train import probes, instruments

MAX_HOURS = 8.0
TURBO = False    # compile gate: KNOWN-FAIL on Blackwell (compiled bf16
                 # backward emits NaN grads — caught 2026-08-15) AND the
                 # trainer must never checkpoint a compiled model
                 # (OptimizedModule key prefix corrupts resume). Leave
                 # False until the NaN hunt lands; eager fused 1.7x is
                 # always on.
REPO = "AbstractPhil/alephllm-mini-beatrix-training"

# continue-safety: this cell is safely RE-RUNNABLE in a live kernel —
# tear down any previous run first so prepare() doesn't stack a second
# model+optimizer into VRAM
import gc
if "run" in globals():
    del run
gc.collect()
torch.cuda.empty_cache()

run = prepare("mini-beatrix-1", hf_token=HF_TOKEN)
added = C.append_curriculum_phases(run.manifest)
if added:
    print(f"curriculum registered: {added} stages appended to the manifest")
if float(run.raw_model.head.gamma.item()) != 1.0:
    info = C.fold_head_gate(run.raw_model)
    print(f"head gate folded: gamma {info['gamma_before']:.6f} -> 1.0 "
          f"(||W_s||={info['w_s_norm']:.4f}; Muon re-elects from here)")

if TURBO:
    import copy as _copy, time as _time
    _vs = build_stream("fineweb-edu", run.tokenizer, run.cfg.context,
                       run.tc.micro_batch, seed=run.tc.seed + 9999,
                       role="val")
    # SMALL probe (4 rows) under bf16 autocast — the gate must test the
    # regime she actually trains in; a full-batch fp32 probe with a live
    # backward graph OOMed a 96GB card (measured 2026-08-15)
    _xb = _vs.next_batch()[:4].to(run.device)

    def _probe(mm, plist):
        for p in plist:
            p.grad = None
        with torch.autocast("cuda", dtype=torch.bfloat16):
            _, loss = mm(_xb[:, :-1], targets=_xb[:, 1:])
        loss.backward()
        g = torch.cat([(p.grad if p.grad is not None
                        else torch.zeros_like(p)).flatten() for p in plist])
        return float(loss.item()), g

    _ref_p = list(run.raw_model.parameters())
    _l0, _g0 = _probe(run.raw_model, _ref_p)
    _g0 = _g0.clone()
    for p in _ref_p:                 # free grads BEFORE the deepcopy —
        p.grad = None                # otherwise the twin copies them too
    torch.cuda.empty_cache()
    _twin = _copy.deepcopy(run.raw_model)
    _twin_p = list(_twin.parameters())
    _twin.compile_hubs()
    _probe(_twin, _twin_p)                      # compile warmup
    _l1, _g1 = _probe(_twin, _twin_p)
    _dl = abs(_l1 - _l0)
    _dg = float((_g1 - _g0).abs().max().item())
    _t0 = _time.time(); [_probe(_twin, _twin_p) for _ in range(3)]
    _tc = (_time.time() - _t0) / 3
    _t0 = _time.time(); [_probe(run.raw_model, _ref_p) for _ in range(3)]
    _te = (_time.time() - _t0) / 3
    if _dl < 5e-3 and _dg < 5e-2:    # bf16-autocast tolerances
        run.raw_model.compile_hubs()
        print(f"[turbo] gate PASSED on this GPU: dloss {_dl:.2e} · "
              f"max|dgrad| {_dg:.2e} · fwd+bwd {_te:.2f}s eager -> "
              f"{_tc:.2f}s compiled ({_te/max(_tc,1e-9):.1f}x)")
    else:
        print(f"[turbo] gate FAILED (dloss {_dl:.2e}, max|dgrad| {_dg:.2e})"
              " — staying on the eager fused path")
    del _twin, _twin_p, _g0, _g1
    for p in _ref_p:
        p.grad = None
    torch.cuda.empty_cache()

def boundary_instruments(run, tag):
    m, tok = run.raw_model, run.tokenizer
    res = probes.run_all(m, tok, run.device)
    vs = build_stream("fineweb-edu", tok, run.cfg.context,
                      run.tc.micro_batch, seed=run.tc.seed + 9999, role="val")
    val = [vs.next_batch().to(run.device) for _ in range(2)]
    led = instruments.toggle_ledger(m, val)
    ws = float(m.head.w_s.weight.norm().item())
    print(f"== boundary {tag} · step {run.step:,} ==")
    print(probes.report(res))
    print(f"  fineweb gauge: full {led['bpb_full']:.4f} · "
          f"head_off {led['toggle_head_aleph_off']:+.4f} · "
          f"hub_off {led.get('toggle_hub_off', 0):+.4f} · ||W_s||={ws:.3f}")
    rep = {"tag": tag, "step": run.step, "probes": res, "ledger": led,
           "w_s_norm": ws}
    name = f"boundary_{tag}_step{run.step}.json"
    with open(name, "w", encoding="utf-8") as f:
        json.dump(rep, f, indent=1, default=float)
    HfApi(token=HF_TOKEN).upload_file(path_or_fileobj=name, repo_id=REPO,
        path_in_repo=f"mini-beatrix-1/reports/curriculum/{name}",
        commit_message=f"curriculum boundary report: {tag}")

t_end = time.time() + MAX_HOURS * 3600
while time.time() < t_end:
    ph = run.manifest.current_phase()
    if ph is None:
        print("curriculum COMPLETE — all stages done."); break
    remaining = int(ph["planned_tokens"]) - int(ph.get("tokens_done", 0))
    hours_left = (t_end - time.time()) / 3600
    print(f"[school] stage {ph['name']} · {remaining/1e9:.3f}B to go · "
          f"session budget {hours_left:.2f}h")
    run.train(max_tokens=remaining, max_hours=hours_left)
    ph_after = run.manifest.current_phase()
    if ph_after is None or ph_after["name"] != ph["name"]:
        boundary_instruments(run, tag=ph["name"])
    else:
        print("[school] session budget reached mid-stage — resume next "
              "session, she keeps her place.")
        break

## Cell 4 — growth report (run anytime)
The childhood so far: every boundary's probe accuracies against the baseline, holdout families marked, head-election trace.

In [ ]:
import json
from huggingface_hub import HfApi, hf_hub_download
REPO = "AbstractPhil/alephllm-mini-beatrix-training"
api = HfApi()
files = [f for f in api.list_repo_files(REPO)
         if f.startswith("mini-beatrix-1/reports/curriculum/")]
reports = []
for f in sorted(files):
    d = json.load(open(hf_hub_download(REPO, f), encoding="utf-8"))
    if "tag" in d:                       # boundary-report schema
        reports.append((d["tag"], d["step"], d["probes"], d.get("w_s_norm")))
    else:                                # baseline/gate schema: {step: {...}}
        kind = "t0" if "baseline" in f else "gate"
        for step, r in d.items():
            reports.append((f"{kind}@{step}", int(step), r["probes"], None))
reports.sort(key=lambda r: r[1])
if not reports:
    print("no reports yet — run Cell 2 (baseline) first")
else:
    suites = sorted(reports[0][2])
    w = max(len(t) for t, *_ in reports) + 2
    print("".ljust(w) + "".join(s.replace("_", " ")[:14].ljust(15)
                                for s in suites))
    for tag, step, pr, ws in reports:
        row = "".join(f"{pr[s]['acc']:.3f}".ljust(15) for s in suites)
        tail = f"  ||W_s||={ws:.3f}" if ws is not None else ""
        print(tag.ljust(w) + row + tail)

## Cell 5 — arm collectives (boundary causal test)
Arrives with geolip 0.6.1 + the amoe harness pass. The holdout families (bAbI 16/19, depth-5 chains, 3-digit subtraction) are **already excluded from every training row**, and the probe batteries already gauge them — so any boundary she passes before this cell lands can be tested retroactively against stored checkpoints (retro-sweep law). Nothing is lost by training first.